# Time Plots and Seasonal Plots

> *"Always start by graphing the data."* — The first rule of time series analysis (FPP3, Hyndman & Athanasopoulos)

Before fitting any model, before computing any statistic, **look at the data**. Time plots, seasonal plots, and subseries plots are the essential visual tools that reveal trend, seasonality, cycles, and anomalies at a glance.

This notebook covers:
1. **Time plots** — basic line plots done right, with proper date formatting
2. **Seasonal plots** — overlaying years to compare seasonal patterns
3. **Seasonal subseries plots** — mini time series per season to see within-season trends
4. **Multi-panel layouts** — subplots for comparing multiple series
5. **Matplotlib date formatting** — locators, formatters, and styling

---
## 1. Setup

In [ ]:
import calendar
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

DATA_DIR = Path("../data")

### Load the datasets

In [ ]:
# Airline passengers (1949-1960) — the classic Box-Jenkins dataset
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.columns = ["Passengers"]
airline.index.freq = "MS"
airline.head()

In [ ]:
# Energy production index (1970-2018)
energy = pd.read_csv(
    DATA_DIR / "EnergyProduction.csv",
    index_col="DATE",
    parse_dates=True,
)
energy.columns = ["EnergyIndex"]
energy.index.freq = "MS"
energy.head()

In [ ]:
# Alcohol sales (1992-2019)
alcohol = pd.read_csv(
    DATA_DIR / "Alcohol_Sales.csv",
    index_col="DATE",
    parse_dates=True,
)
alcohol.columns = ["Sales"]
alcohol.index.freq = "MS"
alcohol.head()

In [ ]:
# Quick shape check
print(f"Airline:  {airline.shape}  |  {airline.index.min()} to {airline.index.max()}")
print(f"Energy:   {energy.shape}  |  {energy.index.min()} to {energy.index.max()}")
print(f"Alcohol:  {alcohol.shape}  |  {alcohol.index.min()} to {alcohol.index.max()}")

---
## 2. Time Plots (Basic Line Plots Done Right)

A **time plot** places time on the x-axis and the observed value on the y-axis. It is the single most important visualization in time series analysis.

### 2.1 The simplest time plot — `df.plot()`

In [ ]:
airline.plot(title="International Airline Passengers (1949-1960)")
plt.ylabel("Thousands of passengers")
plt.xlabel("")
plt.tight_layout()
plt.show()

Even this minimal plot immediately reveals:
- **Upward trend** — passengers grow over time
- **Seasonal pattern** — regular peaks and troughs each year
- **Increasing variance** — the seasonal swings get bigger (multiplicative seasonality)

### 2.2 Proper axis labels, titles, and grids

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(airline.index, airline["Passengers"], color="steelblue", linewidth=1.5)

ax.set_title("International Airline Passengers", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of passengers")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Matplotlib date formatting with locators and formatters

The `matplotlib.dates` module gives fine-grained control over how dates appear on the x-axis:

| Class | Purpose |
|---|---|
| `YearLocator()` | Place a tick at every year (or every *n*th year) |
| `MonthLocator()` | Place a tick at every month (or specific months) |
| `DateFormatter(fmt)` | Format tick labels using `strftime` codes |

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(airline.index, airline["Passengers"], color="steelblue", linewidth=1.5)

# Major ticks at every year, minor ticks at every quarter
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))

ax.set_title("Airline Passengers — Year Ticks with Quarterly Minor Ticks", fontsize=13)
ax.set_ylabel("Thousands of passengers")
ax.grid(which="major", alpha=0.4)
ax.grid(which="minor", alpha=0.15, linestyle="--")

plt.tight_layout()
plt.show()

In [ ]:
# Month-level formatting with rotation — useful for shorter series
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(airline.index, airline["Passengers"], color="steelblue", linewidth=1.5)

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.xticks(rotation=45, ha="right")
ax.set_title("Airline Passengers — Month-Level Formatting", fontsize=13)
ax.set_ylabel("Thousands of passengers")

plt.tight_layout()
plt.show()

### 2.4 Setting x-axis limits to focus on a specific period

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full series
axes[0].plot(airline.index, airline["Passengers"], color="steelblue")
axes[0].set_title("Full Series")
axes[0].set_ylabel("Thousands of passengers")

# Zoomed to 1955-1958
axes[1].plot(airline.index, airline["Passengers"], color="steelblue")
axes[1].set_xlim(pd.Timestamp("1955-01-01"), pd.Timestamp("1958-01-01"))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
axes[1].set_title("Zoomed: 1955–1958")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5 Multiple series on one plot

In [ ]:
# Normalize each series to start at 100 so they can share an axis
airline_norm = airline / airline.iloc[0] * 100
energy_norm = energy / energy.iloc[0] * 100
alcohol_norm = alcohol / alcohol.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(airline_norm.index, airline_norm["Passengers"], label="Airline Passengers", linewidth=1.2)
ax.plot(energy_norm.index, energy_norm["EnergyIndex"], label="Energy Production", linewidth=1.2)
ax.plot(alcohol_norm.index, alcohol_norm["Sales"], label="Alcohol Sales", linewidth=1.2)

ax.set_title("Three Time Series — Indexed to 100 at Start", fontsize=14, fontweight="bold")
ax.set_ylabel("Index (start = 100)")
ax.legend(frameon=True, fancybox=True, shadow=True)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

---
## 3. Seasonal Plots

A **seasonal plot** overlays the same "season" (e.g., month) across different years. Each year becomes a separate line, all plotted against the season axis (1–12 for monthly data). This makes it easy to:

- Identify the **peak** and **trough** months
- See how the seasonal pattern **evolves** over time
- Spot **outlier** years

### 3.1 Building the seasonal plot step by step

In [ ]:
# Step 1: Extract month and year from the DatetimeIndex
air = airline.copy()
air["month"] = air.index.month
air["year"] = air.index.year
air.head()

In [ ]:
# Step 2: Pivot so that each column is one year, rows are months 1-12
pivot = air.pivot(index="month", columns="year", values="Passengers")
pivot

In [ ]:
# Step 3: Plot with a color gradient — older years lighter, newer years darker
fig, ax = plt.subplots(figsize=(12, 6))

years = pivot.columns
cmap = plt.cm.YlOrRd  # light yellow -> dark red
colors = [cmap(i / len(years)) for i in range(len(years))]

for year, color in zip(years, colors):
    ax.plot(
        pivot.index,
        pivot[year],
        marker="o",
        markersize=4,
        linewidth=1.5,
        color=color,
        label=str(year),
    )

ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_title("Seasonal Plot — Airline Passengers", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Thousands of passengers")
ax.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()

**What the seasonal plot reveals:**
- July / August are consistently the peak months (summer travel)
- November is typically the trough
- The color gradient shows that each successive year sits higher — confirming the upward trend
- The gap between years widens toward the peaks — multiplicative seasonality

### 3.2 Seasonal plot for Energy Production

In [ ]:
# Energy seasonal plot — select every 5th year to keep it readable
en = energy.copy()
en["month"] = en.index.month
en["year"] = en.index.year

# Filter to complete years only
year_counts = en.groupby("year").size()
complete_years = year_counts[year_counts == 12].index
en = en[en["year"].isin(complete_years)]

pivot_en = en.pivot(index="month", columns="year", values="EnergyIndex")

# Select a subset of years for clarity
selected_years = [y for y in pivot_en.columns if y % 5 == 0]
pivot_en_sub = pivot_en[selected_years]

fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.viridis
colors = [cmap(i / len(selected_years)) for i in range(len(selected_years))]

for year, color in zip(selected_years, colors):
    ax.plot(
        pivot_en_sub.index, pivot_en_sub[year],
        marker="o", markersize=4, linewidth=1.5,
        color=color, label=str(year),
    )

ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_title("Seasonal Plot — Energy Production (Every 5th Year)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Energy Index")
ax.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

### 3.3 Seasonal plot for Alcohol Sales

In [ ]:
alc = alcohol.copy()
alc["month"] = alc.index.month
alc["year"] = alc.index.year

year_counts_alc = alc.groupby("year").size()
complete_years_alc = year_counts_alc[year_counts_alc == 12].index
alc = alc[alc["year"].isin(complete_years_alc)]

pivot_alc = alc.pivot(index="month", columns="year", values="Sales")

fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.cool
colors = [cmap(i / len(pivot_alc.columns)) for i in range(len(pivot_alc.columns))]

for year, color in zip(pivot_alc.columns, colors):
    ax.plot(
        pivot_alc.index, pivot_alc[year],
        marker=".", markersize=3, linewidth=1.0,
        color=color, label=str(year), alpha=0.8,
    )

ax.set_xticks(range(1, 13))
ax.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)])
ax.set_title("Seasonal Plot — Alcohol Sales", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (millions $)")
ax.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

---
## 4. Seasonal Subseries Plots

A **seasonal subseries plot** creates a mini time series for each season (month). Within each panel:
- The x-axis is the year
- The y-axis is the value for that specific month
- A horizontal line marks the **mean** for that month

This reveals both the **seasonal pattern** (compare means across months) and the **trend within each season** (the slope within each panel).

### 4.1 Subseries plot for Airline Passengers

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 10), sharey=True)
fig.suptitle("Seasonal Subseries Plot — Airline Passengers", fontsize=15, fontweight="bold", y=1.01)

for i, month in enumerate(range(1, 13)):
    ax = axes[i // 3, i % 3]
    monthly_data = airline[airline.index.month == month]
    values = monthly_data["Passengers"].values
    years = monthly_data.index.year

    ax.plot(years, values, marker="o", markersize=5, linewidth=1.2, color="steelblue")
    ax.axhline(values.mean(), color="red", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.set_title(calendar.month_abbr[month], fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

**What the subseries plot reveals:**
- The red dashed lines show that Jul and Aug have the highest monthly means (seasonal pattern)
- Within each panel, the upward slope confirms the growth trend exists **in every month**
- The slope is steepest for summer months — again confirming multiplicative seasonality

### 4.2 Subseries plot for Energy Production

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 10), sharey=True)
fig.suptitle("Seasonal Subseries Plot — Energy Production", fontsize=15, fontweight="bold", y=1.01)

for i, month in enumerate(range(1, 13)):
    ax = axes[i // 3, i % 3]
    monthly_data = energy[energy.index.month == month]
    values = monthly_data["EnergyIndex"].values
    years = monthly_data.index.year

    ax.plot(years, values, marker=".", markersize=3, linewidth=0.8, color="darkorange")
    ax.axhline(values.mean(), color="red", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.set_title(calendar.month_abbr[month], fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", labelsize=7, rotation=45)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

### 4.3 Subseries plot for Alcohol Sales

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(14, 10), sharey=True)
fig.suptitle("Seasonal Subseries Plot — Alcohol Sales", fontsize=15, fontweight="bold", y=1.01)

for i, month in enumerate(range(1, 13)):
    ax = axes[i // 3, i % 3]
    monthly_data = alcohol[alcohol.index.month == month]
    values = monthly_data["Sales"].values
    years = monthly_data.index.year

    ax.plot(years, values, marker=".", markersize=3, linewidth=0.8, color="teal")
    ax.axhline(values.mean(), color="red", linestyle="--", linewidth=1.0, alpha=0.7)
    ax.set_title(calendar.month_abbr[month], fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", labelsize=7, rotation=45)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

---
## 5. Subplots for Multiple Time Series

When comparing datasets, a multi-panel layout keeps each series on its own y-axis while aligning the time axis.

### 5.1 Side-by-side comparison (2x2 grid)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Airline passengers
axes[0, 0].plot(airline.index, airline["Passengers"], color="steelblue")
axes[0, 0].set_title("Airline Passengers", fontweight="bold")
axes[0, 0].set_ylabel("Thousands")

# Energy production
axes[0, 1].plot(energy.index, energy["EnergyIndex"], color="darkorange")
axes[0, 1].set_title("Energy Production", fontweight="bold")
axes[0, 1].set_ylabel("Index")

# Alcohol sales
axes[1, 0].plot(alcohol.index, alcohol["Sales"], color="teal")
axes[1, 0].set_title("Alcohol Sales", fontweight="bold")
axes[1, 0].set_ylabel("Millions $")

# Year-over-year percentage change — airline
pct_change = airline["Passengers"].pct_change(12) * 100
axes[1, 1].plot(pct_change.index, pct_change.values, color="crimson", linewidth=1.0)
axes[1, 1].axhline(0, color="black", linewidth=0.5)
axes[1, 1].set_title("Airline YoY % Change", fontweight="bold")
axes[1, 1].set_ylabel("Percent")

for ax in axes.flat:
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

### 5.2 Stacked subplots sharing the x-axis

In [ ]:
# Focus on a shared time period (1992 onward)
energy_sub = energy.loc["1992":]
alcohol_sub = alcohol.loc["1992":]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(energy_sub.index, energy_sub["EnergyIndex"], color="darkorange")
axes[0].set_ylabel("Energy Index")
axes[0].set_title("Energy Production vs. Alcohol Sales (1992 onward)", fontsize=13, fontweight="bold")

axes[1].plot(alcohol_sub.index, alcohol_sub["Sales"], color="teal")
axes[1].set_ylabel("Alcohol Sales (M$)")

axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Matplotlib Date Formatting Deep Dive

Getting date axes right is a common pain point. Here is a reference for the key tools.

### 6.1 Major and minor locators

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(energy.index, energy["EnergyIndex"], color="darkorange", linewidth=0.8)

# Major ticks every 5 years, minor every year
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_minor_locator(mdates.YearLocator(1))

# Style major vs. minor grid differently
ax.grid(which="major", color="gray", alpha=0.5, linewidth=0.8)
ax.grid(which="minor", color="gray", alpha=0.15, linewidth=0.4, linestyle="--")
ax.tick_params(which="major", length=6)
ax.tick_params(which="minor", length=3)

ax.set_title("Energy Production — Major Ticks Every 5 Years, Minor Every Year", fontsize=13)

plt.tight_layout()
plt.show()

### 6.2 Custom date formatters and label rotation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

subset = airline.loc["1958":"1960"]

# Format A: %Y-%m
axes[0].plot(subset.index, subset["Passengers"], color="steelblue")
axes[0].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
axes[0].set_title("Format: %Y-%m")
axes[0].tick_params(axis="x", rotation=45)

# Format B: %b '%y
axes[1].plot(subset.index, subset["Passengers"], color="steelblue")
axes[1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b '%y"))
axes[1].set_title("Format: %b '%y")
axes[1].tick_params(axis="x", rotation=45)

# Format C: multi-line with \n
axes[2].plot(subset.index, subset["Passengers"], color="steelblue")
axes[2].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
axes[2].set_title("Format: %b (newline) %Y")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.3 Grid styling for time series

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Style 1: Minimal
axes[0].plot(airline.index, airline["Passengers"], color="steelblue")
axes[0].grid(True, alpha=0.15)
axes[0].set_title("Minimal Grid", fontweight="bold")
axes[0].set_ylabel("Passengers")

# Style 2: Emphasized horizontal gridlines (good for reading values)
axes[1].plot(airline.index, airline["Passengers"], color="steelblue")
axes[1].grid(axis="y", alpha=0.4, linewidth=0.8)
axes[1].grid(axis="x", alpha=0.1, linewidth=0.4, linestyle=":")
axes[1].set_title("Horizontal Emphasis", fontweight="bold")
axes[1].set_ylabel("Passengers")

plt.tight_layout()
plt.show()

### 6.4 Annotating key events on a time plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(airline.index, airline["Passengers"], color="steelblue", linewidth=1.5)

# Mark the overall maximum
max_idx = airline["Passengers"].idxmax()
max_val = airline["Passengers"].max()

ax.annotate(
    f"Peak: {max_val:.0f}k\n({max_idx.strftime('%b %Y')})",
    xy=(max_idx, max_val),
    xytext=(max_idx - pd.DateOffset(years=3), max_val + 30),
    fontsize=10,
    arrowprops=dict(arrowstyle="->", color="red", lw=1.5),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray"),
)

# Mark the overall minimum
min_idx = airline["Passengers"].idxmin()
min_val = airline["Passengers"].min()

ax.annotate(
    f"Trough: {min_val:.0f}k\n({min_idx.strftime('%b %Y')})",
    xy=(min_idx, min_val),
    xytext=(min_idx + pd.DateOffset(years=2), min_val - 10),
    fontsize=10,
    arrowprops=dict(arrowstyle="->", color="blue", lw=1.5),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcyan", edgecolor="gray"),
)

ax.set_title("Airline Passengers — Annotated Time Plot", fontsize=14, fontweight="bold")
ax.set_ylabel("Thousands of passengers")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

### 6.5 Shading regions of interest

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(energy.index, energy["EnergyIndex"], color="darkorange", linewidth=1.0)

# Shade recessions (approximate)
recessions = [
    ("1973-11", "1975-03"),
    ("1980-01", "1980-07"),
    ("1981-07", "1982-11"),
    ("1990-07", "1991-03"),
    ("2001-03", "2001-11"),
    ("2007-12", "2009-06"),
]

for start, end in recessions:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.15, color="gray")

ax.set_title("Energy Production with Recession Shading", fontsize=14, fontweight="bold")
ax.set_ylabel("Energy Index")
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# Add legend entry for shading
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor="gray", alpha=0.15, label="US Recession")], loc="upper left")

plt.tight_layout()
plt.show()

---
## 7. Putting It All Together — A Dashboard View

In [ ]:
# Comprehensive dashboard for airline passengers: time plot + seasonal + subseries
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Airline Passengers — Visual Analysis Dashboard", fontsize=16, fontweight="bold", y=0.98)

# Top row: time plot (spans full width)
ax_time = fig.add_subplot(3, 1, 1)
ax_time.plot(airline.index, airline["Passengers"], color="steelblue", linewidth=1.5)
ax_time.set_title("Time Plot", fontsize=12, fontweight="bold")
ax_time.set_ylabel("Passengers (thousands)")
ax_time.xaxis.set_major_locator(mdates.YearLocator())
ax_time.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_time.grid(True, alpha=0.3)

# Bottom-left: seasonal plot
ax_season = fig.add_subplot(3, 2, 5)
pivot_air = air.pivot(index="month", columns="year", values="Passengers")
cmap = plt.cm.YlOrRd
for idx, year in enumerate(pivot_air.columns):
    color = cmap(idx / len(pivot_air.columns))
    ax_season.plot(pivot_air.index, pivot_air[year], color=color, linewidth=1.0, alpha=0.8)
ax_season.set_xticks(range(1, 13))
ax_season.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)], fontsize=8)
ax_season.set_title("Seasonal Plot", fontsize=12, fontweight="bold")
ax_season.set_ylabel("Passengers")
ax_season.grid(True, alpha=0.3)

# Bottom-right: monthly means bar chart (summary of seasonal pattern)
ax_bar = fig.add_subplot(3, 2, 6)
monthly_means = airline.groupby(airline.index.month)["Passengers"].mean()
bars = ax_bar.bar(
    range(1, 13), monthly_means.values,
    color=["steelblue" if v < monthly_means.mean() else "coral" for v in monthly_means.values],
    edgecolor="white", linewidth=0.5,
)
ax_bar.axhline(monthly_means.mean(), color="red", linestyle="--", alpha=0.6, label="Overall mean")
ax_bar.set_xticks(range(1, 13))
ax_bar.set_xticklabels([calendar.month_abbr[m] for m in range(1, 13)], fontsize=8)
ax_bar.set_title("Monthly Averages", fontsize=12, fontweight="bold")
ax_bar.set_ylabel("Mean passengers")
ax_bar.legend(fontsize=9)
ax_bar.grid(axis="y", alpha=0.3)

# Middle row: year-over-year growth
ax_growth = fig.add_subplot(3, 1, 2)
yoy = airline["Passengers"].pct_change(12) * 100
ax_growth.bar(yoy.index, yoy.values, width=25, color=np.where(yoy.values >= 0, "seagreen", "tomato"))
ax_growth.axhline(0, color="black", linewidth=0.5)
ax_growth.set_title("Year-over-Year Growth (%)", fontsize=12, fontweight="bold")
ax_growth.set_ylabel("% change")
ax_growth.xaxis.set_major_locator(mdates.YearLocator())
ax_growth.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax_growth.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Summary

| Plot | What it shows | When to use |
|---|---|---|
| **Time plot** | Overall trend, seasonality, cycles, outliers | Always — it is the **first** thing you do |
| **Seasonal plot** | Seasonal pattern overlaid across years | When you suspect seasonality and want to see how it evolves |
| **Subseries plot** | Trend within each season + seasonal pattern via means | When you want both seasonal pattern AND within-season trend |
| **Multi-panel subplot** | Comparison of multiple series | When analyzing related series side by side |

### Key takeaways

1. **Time plot = first thing you do.** Before any model, before any statistic — graph the data.
2. **Seasonal plot = see patterns across years.** Overlay years on a month axis to reveal the seasonal shape and how it changes.
3. **Subseries plot = see trends within each season.** Each month gets its own mini time series, with a mean line for reference.
4. **Date formatting matters.** Use `mdates` locators and formatters for clean, professional axes.
5. **Annotations and shading** add context — mark peaks, troughs, recessions, or regime changes.